# Analysis of Synthetic Data Generation Results
This notebook processes, merges, and visualizes benchmark results from `Results/gen_params_XX` (or `gen_params_06` / `gen_params_07`).

### Research Questions Covered:
1. **RQ1 (Baseline Performance):** How do ML models perform across different initial seed sizes (1 to 20) before and after hyperparameter tuning?
2. **RQ2 (Generator vs Model Matrix):** How does each synthetic data generation method impact different ML classifiers (paired difference vs. Baseline and vs. Tuned Baseline)?
3. **RQ3 (Scaling Dynamics):** How does expanding synthetic data volume from 1x (seed) up to 10x affect accuracy across seed sizes?
4. **RQ4 (Seed Size Trajectories):** For each generator, how does initial training seed volume (1, 2, 5, 10, 20) influence scaling response?
5. **RQ5 (Parameter Sensitivity Analysis):** Which generation hyperparameter sets perform best, and how sensitive is each generator to parameter tuning?
6. **RQ6 (Statistical Significance):** Are the performance gains statistically significant (Wilcoxon signed-rank test, p < 0.05)?
7. **RQ7 (Efficiency & Computational Cost):** Execution time vs. accuracy gain trade-off across generators.


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set visual aesthetic
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 16
})

# Choose the run folder to analyze ('gen_params_06' or future 'gen_params_07')
run_folder = 'gen_params_06'
output_dir = f'Output_Figures/{run_folder}'
os.makedirs(output_dir, exist_ok=True)
print(f"Target Run Folder: {run_folder}")
print(f"Output Directory : {output_dir}")

# CANONICAL UNIFIED ORDERING of generative methods across ALL figures (starting from Jittering)
CANONICAL_METHODS = [
    'Jittering',
    'Scaling',
    'Magnitude_Warping',
    'Time_Warping',
    'SMOTE',
    'ADASYN',
    'Mixup',
    'GMM',
    'HMM_GMM',
    'TimeVAE'
]

# Permanent colors and markers mapped to each method across the entire notebook
colors = sns.color_palette('tab10', len(CANONICAL_METHODS))
fixed_palette = dict(zip(CANONICAL_METHODS, colors))
fixed_markers = dict(zip(CANONICAL_METHODS, ['o', 's', '^', 'v', 'D', 'p', '*', 'X', '<', '>']))


## 1. Merge & Clean Benchmark Results
Reads all `*_results.csv` files, handles header rows, aggregates across the 10 seed fold selections, and preserves both **Baseline**, **Tuned_Baseline**, and **Generator** records.


In [ ]:
result_dir = f'../Results/{run_folder}'
all_csv_files = glob.glob(os.path.join(result_dir, '*_results.csv'))

print(f"Found {len(all_csv_files)} CSV files in {result_dir}.")

expected_cols = [
    'Dataset', 'Fold', 'SeedsPerLabel', 'SeedFoldSelection', 'ScaleFactor', 
    'Model', 'Method', 'GenParamSetIndex', 'GenParams', 'Accuracy', 
    'F1_Score', 'Hyperparameters', 'ExecutionTime_Sec'
]

df_list = []
for file in all_csv_files:
    try:
        df = pd.read_csv(file, names=expected_cols, header=None, on_bad_lines='skip', low_memory=False)
        df_list.append(df)
    except Exception as e:
        print(f"Error reading {file}: {e}")

df_raw = pd.concat(df_list, ignore_index=True)

# Drop any accidental repeated header rows
df_raw = df_raw[df_raw['Dataset'] != 'Dataset']

# Convert numerical columns
for col in ['Accuracy', 'F1_Score', 'ExecutionTime_Sec', 'Fold', 'SeedsPerLabel', 'ScaleFactor', 'GenParamSetIndex']:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Drop invalid accuracy rows
df_raw = df_raw.dropna(subset=['Accuracy'])

# Handle GenParamSetIndex and GenParams for Baseline/Tuned_Baseline rows to prevent dropping during groupby
df_raw['GenParamSetIndex'] = df_raw['GenParamSetIndex'].fillna(-1).astype(int)
df_raw['GenParams'] = df_raw['GenParams'].fillna('N/A').astype(str)

print(f"Total valid raw rows loaded: {len(df_raw):,}")

# Aggregate across the 10 random SeedFoldSelections for stable evaluation
groupby_cols = ['Dataset', 'Fold', 'SeedsPerLabel', 'ScaleFactor', 'Model', 'Method', 'GenParamSetIndex', 'GenParams']
df_raw['Hyperparameters'] = df_raw['Hyperparameters'].astype(str)
df_agg = df_raw.groupby(groupby_cols, as_index=False).agg({
    'Accuracy': 'mean',
    'F1_Score': 'mean',
    'ExecutionTime_Sec': 'mean',
    'Hyperparameters': 'first'
})

print(f"Aggregated rows across SeedFoldSelections: {len(df_agg):,}")

# Separate baseline records and synthetic generator records
df_baseline = df_agg[df_agg['Method'].isin(['Baseline', 'Tuned_Baseline'])].copy()
df_gen = df_agg[~df_agg['Method'].isin(['Baseline', 'Tuned_Baseline'])].copy()

# For RQ1-RQ4: Identify the best parameter set (GenParamSetIndex) for each generator per (Dataset, Model, SeedsPerLabel, ScaleFactor)
mean_f1 = df_gen.groupby(['Dataset', 'Model', 'SeedsPerLabel', 'ScaleFactor', 'Method', 'GenParamSetIndex'])['F1_Score'].mean().reset_index()
best_params_idx = mean_f1.loc[mean_f1.groupby(['Dataset', 'Model', 'SeedsPerLabel', 'ScaleFactor', 'Method'])['F1_Score'].idxmax()]
df_gen_best = pd.merge(df_gen, best_params_idx[['Dataset', 'Model', 'SeedsPerLabel', 'ScaleFactor', 'Method', 'GenParamSetIndex']], 
                       on=['Dataset', 'Model', 'SeedsPerLabel', 'ScaleFactor', 'Method', 'GenParamSetIndex'], how='inner')

# Combine baseline + best generator results for visualization
df_all = pd.concat([df_baseline, df_gen_best], ignore_index=True)

# Save merged files
df_all.to_csv(f'{output_dir}/Merged_Results.csv', index=False)
print(f"Saved best-parameter merged results to '{output_dir}/Merged_Results.csv' ({len(df_all):,} rows).")

# Save all parameter sets for sensitivity analysis (RQ5)
df_gen.to_csv(f'{output_dir}/Merged_All_GenParams.csv', index=False)
print(f"Saved all generator parameter records to '{output_dir}/Merged_All_GenParams.csv' ({len(df_gen):,} rows).")

display(df_all.head())


## Research Question 1: Baseline Performance & Tuning Gains
Compares raw `Baseline` vs. `Tuned_Baseline` across initial seed sizes (1, 2, 5, 10, 20) for all ML models and individual datasets.


In [ ]:
baseline_agg = df_baseline.groupby(['Dataset', 'Model', 'SeedsPerLabel', 'Method'])['Accuracy'].mean().reset_index()
datasets = sorted(baseline_agg['Dataset'].unique())
model_order = [m for m in ['KNN', 'LinearSVC', 'LogisticRegression', 'RandomForest', 'RNN'] if m in baseline_agg['Model'].unique()]

print("=== Research Question 1: Baseline Performance per Dataset ===")
for dataset in datasets:
    df_ds = baseline_agg[baseline_agg['Dataset'] == dataset]
    g = sns.catplot(
        data=df_ds, x='Model', y='Accuracy', hue='Method', col='SeedsPerLabel', 
        order=model_order, kind='bar', palette='viridis', height=4.2, aspect=1.05
    )
    # Rotate model names and align right to completely prevent overlapping
    g.set_xticklabels(rotation=35, ha='right', fontsize=10.5)
    g.figure.suptitle(f"Baseline Accuracy per Model and Seed Size - Dataset: {dataset}", y=1.06, fontsize=15, weight='bold')
    plt.savefig(f'{output_dir}/RQ1_Baseline_{dataset}.png', dpi=300, bbox_inches='tight')
    plt.show()

print("=== Average Baseline Performance Across All 5 Datasets ===")
overall_baseline = baseline_agg.groupby(['Model', 'SeedsPerLabel', 'Method'])['Accuracy'].mean().reset_index()

g = sns.catplot(
    data=overall_baseline, x='Model', y='Accuracy', hue='Method', col='SeedsPerLabel', 
    order=model_order, kind='bar', palette='viridis', height=4.2, aspect=1.05
)
# Rotate model names and align right to completely prevent overlapping
g.set_xticklabels(rotation=35, ha='right', fontsize=10.5)
g.figure.suptitle("Average Baseline Accuracy Across All Datasets", y=1.06, fontsize=15, weight='bold')
plt.savefig(f'{output_dir}/RQ1_Baseline_Overall.png', dpi=300, bbox_inches='tight')
plt.show()


## Research Question 2: Effect of Synthetic Data on Machine Learning Algorithms
Calculates paired differences for each (Dataset, Fold, Model, SeedsPerLabel):
1. **Accuracy Gain vs. Default Baseline:** $\Delta Acc = Acc_{syn} - Acc_{baseline}$
2. **Accuracy Gain vs. Tuned Baseline:** $\Delta Acc = Acc_{syn} - Acc_{tuned\_baseline}$ (reveals whether data generation beats model hyperparameter tuning).


In [ ]:
# Lookup reference baseline and tuned baseline
base_lookup = df_baseline[df_baseline['Method'] == 'Baseline'].rename(columns={'Accuracy': 'Baseline_Acc'})[['Dataset', 'Model', 'SeedsPerLabel', 'Fold', 'Baseline_Acc']]
tuned_lookup = df_baseline[df_baseline['Method'] == 'Tuned_Baseline'].rename(columns={'Accuracy': 'Tuned_Acc'})[['Dataset', 'Model', 'SeedsPerLabel', 'Fold', 'Tuned_Acc']]

df_gen_paired = pd.merge(df_gen_best, base_lookup, on=['Dataset', 'Model', 'SeedsPerLabel', 'Fold'], how='inner')
df_gen_paired = pd.merge(df_gen_paired, tuned_lookup, on=['Dataset', 'Model', 'SeedsPerLabel', 'Fold'], how='left')

df_gen_paired['Diff_vs_Baseline'] = df_gen_paired['Accuracy'] - df_gen_paired['Baseline_Acc']
df_gen_paired['Diff_vs_Tuned'] = df_gen_paired['Accuracy'] - df_gen_paired['Tuned_Acc']

# Pivot tables ordered strictly by CANONICAL_METHODS
pivot_vs_base = df_gen_paired.pivot_table(index='Model', columns='Method', values='Diff_vs_Baseline', aggfunc='mean')
pivot_vs_tuned = df_gen_paired.pivot_table(index='Model', columns='Method', values='Diff_vs_Tuned', aggfunc='mean')

valid_cols = [m for m in CANONICAL_METHODS if m in pivot_vs_base.columns]
pivot_vs_base = pivot_vs_base.reindex(columns=valid_cols)
pivot_vs_tuned = pivot_vs_tuned.reindex(columns=valid_cols)

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

sns.heatmap(pivot_vs_base, annot=True, fmt='+.3f', cmap='coolwarm', center=0, cbar_kws={'label': 'Mean Accuracy Gain'}, ax=axes[0])
axes[0].set_title("RQ2.1: Accuracy Gain vs. Default Baseline (Ordered: Jittering -> TimeVAE)", fontsize=14, weight='bold')
axes[0].set_xlabel("")

sns.heatmap(pivot_vs_tuned, annot=True, fmt='+.3f', cmap='coolwarm', center=0, cbar_kws={'label': 'Mean Accuracy Gain'}, ax=axes[1])
axes[1].set_title("RQ2.2: Accuracy Gain vs. Tuned Baseline (Net Benefit Over HP Tuning)", fontsize=14, weight='bold')
axes[1].set_xlabel("Generation Method", fontsize=12)

plt.tight_layout()
plt.savefig(f'{output_dir}/RQ2_Heatmap_Accuracy_Change.png', dpi=300, bbox_inches='tight')
plt.show()

print("--- Summary vs Default Baseline ---")
display(pivot_vs_base.round(4))
print("--- Summary vs Tuned Baseline ---")
display(pivot_vs_tuned.round(4))


## Data Preparation for Scaling Trajectories (RQ3 & RQ4)
Anchor synthetic methods to `ScaleFactor = 1` using the baseline measurement, attaching only to generator methods that actually exist for each seed size.


In [ ]:
df_baseline_start = df_baseline[df_baseline['Method'] == 'Baseline'].copy()
df_baseline_start['ScaleFactor'] = 1

# Only attach ScaleFactor 1 to methods actually available for that (Model, Dataset, SeedsPerLabel)
valid_combos = df_gen_best[['Model', 'Method', 'SeedsPerLabel', 'Dataset', 'Fold']].drop_duplicates()
df_baseline_anchors = pd.merge(
    valid_combos, 
    df_baseline_start.drop(columns=['Method']), 
    on=['Model', 'Dataset', 'SeedsPerLabel', 'Fold']
)

df_trajectory = pd.concat([df_baseline_anchors, df_gen_best], ignore_index=True)
df_trajectory_agg = df_trajectory.groupby(['Model', 'Method', 'SeedsPerLabel', 'ScaleFactor'])['Accuracy'].mean().reset_index()

print("Trajectory data prepared successfully. Unique scale factors:", sorted(df_trajectory_agg['ScaleFactor'].unique()))


## Research Question 3: Scaling Effect by Seed Size
Examines the effect of scaling factor (1x to 10x) across different seed sizes (1 to 20), comparing trajectories against `Baseline` (gray dashed) and `Tuned_Baseline` (red dash-dot).


In [ ]:
models = sorted(df_trajectory_agg['Model'].unique())
seed_sizes = sorted(df_trajectory_agg['SeedsPerLabel'].unique())

for model in models:
    df_model = df_trajectory_agg[df_trajectory_agg['Model'] == model]
    
    fig, axes = plt.subplots(1, len(seed_sizes), figsize=(4.8 * len(seed_sizes), 5.5), sharey=True)
    fig.suptitle(f"Model: {model} - Scaling Effect by Seed Size", fontsize=16, y=1.04)
    
    if len(seed_sizes) == 1:
        axes = [axes]
        
    for i, seed in enumerate(seed_sizes):
        df_seed = df_model[df_model['SeedsPerLabel'] == seed]
        ax = axes[i]
        
        # Enforce fixed palette, markers, and canonical order so colors NEVER get mixed up between subplots!
        sns.lineplot(
            data=df_seed, x='ScaleFactor', y='Accuracy', 
            hue='Method', style='Method', 
            hue_order=CANONICAL_METHODS, style_order=CANONICAL_METHODS,
            palette=fixed_palette, markers=fixed_markers, dashes=False, ax=ax
        )
        
        # Horizontal lines for baseline performance
        base_acc = df_baseline[(df_baseline['Model'] == model) & (df_baseline['SeedsPerLabel'] == seed) & (df_baseline['Method'] == 'Baseline')]['Accuracy'].mean()
        tuned_acc = df_baseline[(df_baseline['Model'] == model) & (df_baseline['SeedsPerLabel'] == seed) & (df_baseline['Method'] == 'Tuned_Baseline')]['Accuracy'].mean()
        
        if pd.notna(base_acc):
            ax.axhline(base_acc, color='gray', linestyle='--', linewidth=1.5, label='Baseline')
        if pd.notna(tuned_acc):
            ax.axhline(tuned_acc, color='red', linestyle='-.', linewidth=1.5, label='Tuned_Baseline')
            
        ax.set_title(f"Seed Size: {seed}", fontsize=13, weight='bold')
        ax.set_xlabel("Scaling Factor", fontsize=12)
        ax.set_ylabel("Accuracy" if i == 0 else "")
        ax.set_xticks([1, 2, 3, 5, 10])
        
        # Remove individual subplot legends
        if ax.get_legend() is not None:
            ax.get_legend().remove()
            
    # Create one single, authoritative, truthful figure-level legend strictly following CANONICAL_METHODS
    handles = []
    labels = []
    handles.append(plt.Line2D([0], [0], color='gray', linestyle='--', linewidth=1.5))
    labels.append('Baseline')
    handles.append(plt.Line2D([0], [0], color='red', linestyle='-.', linewidth=1.5))
    labels.append('Tuned_Baseline')
    
    for m in CANONICAL_METHODS:
        handles.append(plt.Line2D([0], [0], color=fixed_palette[m], marker=fixed_markers[m], linewidth=1.5, markersize=7))
        labels.append(m)
        
    fig.legend(handles, labels, bbox_to_anchor=(1.02, 0.88), loc='upper left', frameon=True, title="Method", fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{output_dir}/RQ3_{model}_Scaling_SeedSize.png', dpi=300, bbox_inches='tight')
    plt.show()


## Research Question 4: Effect of Seed Size per Generator Method
Shows how initial seed volume (1, 2, 5, 10, 20) determines the trajectory of each data generator from 1x to 10x.


In [ ]:
# Strictly follow CANONICAL_METHODS order starting from Jittering
gen_methods = [m for m in CANONICAL_METHODS if m in df_gen_best['Method'].unique()]
hue_order_rq4 = [str(s) for s in seed_sizes]

for model in models:
    df_model = df_trajectory_agg[df_trajectory_agg['Model'] == model]
    
    fig, axes = plt.subplots(1, len(gen_methods), figsize=(4.2 * len(gen_methods), 5.2), sharey=True)
    fig.suptitle(f"Model: {model} - Effect of Seed Size per Gen Method (Ordered: Jittering -> TimeVAE)", fontsize=16, y=1.04)
    
    if len(gen_methods) == 1:
        axes = [axes]
        
    for i, gen_method in enumerate(gen_methods):
        df_gen_m = df_model[df_model['Method'] == gen_method].copy()
        df_gen_m['SeedsPerLabel'] = df_gen_m['SeedsPerLabel'].astype(str)
        ax = axes[i]
        
        sns.lineplot(
            data=df_gen_m, x='ScaleFactor', y='Accuracy', hue='SeedsPerLabel', style='SeedsPerLabel', 
            hue_order=hue_order_rq4, style_order=hue_order_rq4, palette='tab10', 
            markers=True, dashes=False, ax=ax
        )
        
        ax.set_title(f"{gen_method}", fontsize=13, weight='bold')
        ax.set_xlabel("Scaling Factor", fontsize=12)
        ax.set_ylabel("Accuracy" if i == 0 else "")
        ax.set_xticks([1, 2, 3, 5, 10])
        
        if i < len(gen_methods) - 1:
            if ax.get_legend() is not None:
                ax.get_legend().remove()
        else:
            ax.legend(title='Seed Size', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)
            
    plt.tight_layout()
    plt.savefig(f'{output_dir}/RQ4_{model}_SeedSize_GenMethod.png', dpi=300, bbox_inches='tight')
    plt.show()


## Research Question 5 (New Insight): Generator Hyperparameter Sensitivity
Analyzes the variance in performance across the 10 hyperparameter configurations tested for each generator. This demonstrates whether generator tuning is critical to benchmarking success.


In [ ]:
# Calculate performance spread across GenParamSetIndex for each generator
gen_param_perf = df_gen.groupby(['Method', 'GenParamSetIndex', 'GenParams'])['Accuracy'].agg(['mean', 'std', 'min', 'max']).reset_index()

# Plot spread across parameter sets strictly following CANONICAL_METHODS order
valid_gen_methods = [m for m in CANONICAL_METHODS if m in df_gen['Method'].unique()]

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(data=df_gen, x='Method', y='Accuracy', order=valid_gen_methods, palette=fixed_palette, ax=ax)
ax.set_title("Performance Distribution Across Generation Hyperparameters per Method", fontsize=15, weight='bold')
ax.set_xlabel("Generator Method (Ordered: Jittering -> TimeVAE)", fontsize=12)
ax.set_ylabel("Accuracy", fontsize=12)
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(f'{output_dir}/RQ5_GenParams_Sensitivity_Boxplot.png', dpi=300, bbox_inches='tight')
plt.show()

# Top performing parameter set for each method ordered strictly by CANONICAL_METHODS
top_params = gen_param_perf.sort_values(['Method', 'mean'], ascending=[True, False]).groupby('Method').first().reset_index()
top_params['Method'] = pd.Categorical(top_params['Method'], categories=CANONICAL_METHODS, ordered=True)
top_params = top_params.sort_values('Method').reset_index(drop=True)

print("=== Top Overall Parameter Configuration per Generator (Unified Order) ===")
display(top_params[['Method', 'GenParamSetIndex', 'GenParams', 'mean', 'std']].rename(columns={'mean': 'Overall_Mean_Acc', 'std': 'Std_Acc'}))


## Research Question 6 (New Insight): Statistical Significance Testing
Performs Wilcoxon signed-rank tests between **Best Synthetic Data** vs. **Tuned Baseline** across all folds to verify if accuracy improvements are statistically significant ($p < 0.05$).


In [ ]:
stat_results = []
valid_gen_methods = [m for m in CANONICAL_METHODS if m in df_gen_paired['Method'].unique()]

for model in models:
    for method in valid_gen_methods:
        sub = df_gen_paired[(df_gen_paired['Model'] == model) & (df_gen_paired['Method'] == method)].dropna(subset=['Diff_vs_Tuned'])
        if len(sub) >= 20 and sub['Diff_vs_Tuned'].abs().sum() > 0:
            stat, p_val = stats.wilcoxon(sub['Diff_vs_Tuned'])
            mean_gain = sub['Diff_vs_Tuned'].mean()
            win_rate = (sub['Diff_vs_Tuned'] > 0).mean()
            stat_results.append({
                'Model': model,
                'Method': method,
                'Mean_Gain_vs_Tuned': mean_gain,
                'Win_Rate': win_rate,
                'p_value': p_val,
                'Significant (p < 0.05)': 'Yes' if p_val < 0.05 else 'No'
            })

df_stat = pd.DataFrame(stat_results)
pivot_pvals = df_stat.pivot_table(index='Model', columns='Method', values='p_value')
pivot_sig = df_stat.pivot_table(index='Model', columns='Method', values='Significant (p < 0.05)', aggfunc='first')

pivot_pvals = pivot_pvals.reindex(columns=[m for m in CANONICAL_METHODS if m in pivot_pvals.columns])
pivot_sig = pivot_sig.reindex(columns=[m for m in CANONICAL_METHODS if m in pivot_sig.columns])

print("=== Wilcoxon Test p-values vs. Tuned Baseline (Ordered: Jittering -> TimeVAE) ===")
display(pivot_pvals.round(4))
print("=== Statistically Significant Improvements (p < 0.05) ===")
display(pivot_sig)


## Research Question 7 (New Insight): Computational Cost vs. Accuracy Gain
Evaluates the efficiency trade-off: generation and training execution time vs. the accuracy gain achieved.


In [ ]:
time_gain = df_gen_paired.groupby('Method').agg({
    'ExecutionTime_Sec': 'mean',
    'Diff_vs_Tuned': 'mean'
}).reset_index()

time_gain['Method'] = pd.Categorical(time_gain['Method'], categories=CANONICAL_METHODS, ordered=True)
time_gain = time_gain.sort_values('Method').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=time_gain, x='ExecutionTime_Sec', y='Diff_vs_Tuned', hue='Method', 
    hue_order=[m for m in CANONICAL_METHODS if m in time_gain['Method'].values],
    palette=fixed_palette,
    s=220, ax=ax
)

for _, row in time_gain.iterrows():
    ax.annotate(
        row['Method'], 
        (row['ExecutionTime_Sec'], row['Diff_vs_Tuned']),
        textcoords="offset points", xytext=(0, 10), ha='center', fontsize=11, weight='bold'
    )

ax.axhline(0, color='gray', linestyle='--')
ax.set_title("Efficiency Frontier: Execution Time vs. Accuracy Gain (over Tuned Baseline)", fontsize=14, weight='bold')
ax.set_xlabel("Mean Execution Time per Fold (Seconds)", fontsize=12)
ax.set_ylabel("Mean Accuracy Gain vs. Tuned Baseline", fontsize=12)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f'{output_dir}/RQ7_Efficiency_Tradeoff.png', dpi=300, bbox_inches='tight')
plt.show()
